# DUEL: a from-scratch example

This notebook walks through the two core perplexities reported in the [DUEL paper](https://arxiv.org/abs/2603.01367) on a small piece of text:

1. **ELBO perplexity** -- the standard variational *upper bound* on `-log p(x_0)` used by every MDM (MDLM, SEDD, BD3-LM).
2. **DUEL perplexity** -- the *exact* log-likelihood you get by pairing the same denoiser with a deterministic unmasking policy.

We deliberately bypass `main.py` / Lightning / Hydra and rewrite the algorithms by hand. This makes the math obvious; for real evaluations use `scripts/` instead.

Requirements: a CUDA GPU, the `duel` conda env, and `flash-attn==2.5.6` (needed by the MDLM-OWT HF checkpoint -- see README).

In [70]:
import math
import torch
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 1. Load the denoiser

We use **MDLM-OWT** (`kuleshov-group/mdlm-owt`), an HF DiT that extends the GPT-2 vocabulary by one extra `[MASK]` token at index `vocab_size`.

> **Why MDLM and not [BD3-LM](https://arxiv.org/abs/2503.09573)?** BD3-LM's ELBO uses a *per-block* noise schedule and a 2×-length input (the noised block concatenated with the clean prefix for cross-attention). Reproducing it from scratch would obscure the algorithm. The DUEL framework itself applies to both models — see `scripts/owt_perplexity/bd3lm_*` for the production BD3-LM eval.

In [33]:
denoiser = AutoModelForMaskedLM.from_pretrained(
    "kuleshov-group/mdlm-owt", trust_remote_code=True
).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("gpt2")

VOCAB_SIZE = tokenizer.vocab_size      # 50257 real tokens
MASK_ID = VOCAB_SIZE                   # model's [MASK] token sits at index vocab_size

def denoise(xt: torch.Tensor) -> torch.Tensor:
    """Return log p_theta(x_0 | x_t) over the V real tokens; shape [B, L, V].

    Two HF quirks are handled here, once:
      1. The DiT requires a `timesteps` arg; under SUBS it's just zeros.
      2. The DiT outputs V+1 logits per position -- the extra column is the
         [MASK] token, which is never a valid prediction. We slice it off
         before log-softmax to get a clean distribution over real tokens.
    """
    timesteps = torch.zeros(xt.shape[0], device=xt.device)
    logits = denoiser(xt, timesteps=timesteps)[..., :VOCAB_SIZE]   # [B, L, V]
    return F.log_softmax(logits, dim=-1)                                  # [B, L, V]

# Sanity check.
with torch.no_grad():
    dummy = torch.zeros(1, 4, dtype=torch.long, device=device)
    print(f"log-probs shape: {tuple(denoise(dummy).shape)}  (expected [1, 4, {VOCAB_SIZE}])")

log-probs shape: (1, 4, 50257)  (expected [1, 4, 50257])


## 2. Tokenize the example sentence

The MDLM-OWT dataloader (`dataloader.py:279`) packs every training sequence as `[BOS] tok1 ... tokN [EOS]`. For the GPT-2 tokenizer, both BOS and EOS are `<|endoftext|>` (id 50256). We evaluate on a text snippet taken from the validation set of the OWT dataset, which is used to evaluate the MDLM-OWT checkpoint:

In [71]:
text = """Freaknik: The Musical is an American musical television special produced by T-Pain. It features the voice of T-Pain as the Ghost of Freaknik, as well as the voices of entertainers such as Lil Wayne, Young Cash, Snoop Dogg, Sophia Fresh, and Rick Ross, and comedians such as Andy Samberg and Charlie Murphy who provide additional voices. It was scheduled to air on Cartoon Network's late night programming block Adult Swim sometime in 2009, but after several push-backs, it premiered on March 7, 2010. The musical is based on the actual music festival of the same name that used to take place in Atlanta, Georgia.  A soundtrack was released by Jive and Nappy Boy on April 20, 2010. The 50-minute uncut version of Freaknik: The Musical has been released on DVD and other forms of home media.  Plot [ edit ]  The movie starts at a party that a group of young adults (Christopher "Kid" Reid and Affion Crockett) claims is the best party they have ever attended. An elderly man (Lil Jon) joins the party and explains the history of Freaknik. He tells them that Freaknik threw the biggest party of all time, until it was broken up by the police in 1998; he claims the police "killed" Freaknik. Kid n' Play tried to convince that Freaknik is an urban legend like Candyman, but as Play looks in the mirror, they were eaten by a swarm of wasps. The group is then led by the old man in summoning Freaknik, who appears as the Ghost of Freaknik Past (T-Pain).  The scene changes to a radio announcer named Mr. Thanksgiving (DJ Drama), who is interviewing Freaknik. Mr. Thanksgiving and Freaknik explain that a rapping contest will be held, the victory of which will get "a lifetime supply of money, clothes, and hoes". The scene changes once more to the bedroom of Virgil (Young Cash), Big Uzi (Rick Ross), and Light Skin (CeeLo Green), collectively known as the Sweet Tea Mobsters, a group of young adult rappers from Sweet Tea, Florida, who hope to achieve fame. The group decides to drive to Atlanta to participate in the aforementioned contest, along with their weed-smoking (and supplying) friend Doela Man (DJ Pooh).  During their journey, Light Skin tells of a secret society of African Americans called the Boule, fraternity parlance for "a council of noblemen", that seeks to guide the course of black culture. The members of this organization are parodies of Oprah Winfrey, Al Sharpton (Charlie Murphy), Bill Cosby (Kel Mitchell), Russell Simmons (Affion Crockett), O. J. Simpson, and Jesse Jackson. They wear medallions inscribed "10%", an allusion to the W. E. B. Du Bois essay The Talented Tenth, which says that a class of exceptional members of the black race will rise to lead it.  The Sweet Tea Mobsters make a number of pit stops, including a detour at a college fraternity party where they meet two alcoholic fraternity members (Bill Hader and Andy Samberg). While Virgil is fueling the gas tank, A car, inhabited by four sexy-looking women (Sophia Fresh) named Leacosia (Crystal), Toprameneesha (Skye), Obamaniqua (Cole Rose), and Suzie (Crystal), arrives. He tried to refuse, but the girls beg in song, besides Leacosia giving Virgil a kiss in the form of a gun. He accepts.  Meanwhile, Freaknik meets Rev. Sharpton, who tried to force him to work for the Boule. The plan fails as Freaknik says that he will "never, ever turn his back on his own people", so he escorts Sharpton out via trapdoor.  At the party, the group meets the Fruit Bowl Boys (Kel Mitchell, Affion Crockett, and Denzel Whitaker), who later become the group's biggest competition and are from the mostly white suburbs of Sweet Tea, Florida (although they resemble the Sweet Tea Mobsters). On their long, winding road trip, the Sweet Tea Mob gets lost in New Orleans and are confronted by a gangster (Snoop Dogg) who makes them visit his boss, Trap Jesus (Lil Wayne). Upon meeting Trap Jesus, the group loses hope, thinking it is the end, but instead he inspires them to compete and gives them one of his many Lamborghinis to use to get to Atlanta. However, they crash the Lamborghini when Big Uzi becomes enraged after hearing the Fruit Bowl Boys talking about his jail experience. The group gives up except Virgil, who believes that winning the contest is their destiny. The rest of the group still doesn't believe him until they are given a ride in an airplane by the "Flying Malcolms."  Meanwhile, the Freaknik character is elected the "ghost mayor of Atlanta" and dubs the city "Freaknation." Soon after, President Barack Obama hands the presidency over to the ghost of Freaknik, a move that greatly angers Oprah, which wants to see Freaknik destroyed. She devises a plan to send a giant robotic monster called the "Perminator" (a robotic version of Al Sharpton, rebuilt from Sharpton's corpse after he got hit by lightning while blowing out his hair) to Atlanta to destroy Freaknik. Meanwhile, at the party, the Fruit Bowl Boys begin singing "Shank Ya in the Shower." The Sweet Tea Mobsters arrive at Atlanta at the same time as the Perminator begins its attack; it kills the Fruit Bowl Boys almost immediately. He seems to have Freaknik down for the count, but mass love from the crowd empowers Freaknik as the Mob performs, giving him the ability to grow to a monstrous size. Using the love of his fans, Freaknik is able to destroy the Perminator.  After the fight, Freaknik declares the Sweet Tea Mobsters the winners of the contest, but Virgil refuses the prize and tears the check in half. He tells Freaknik that he doesn't need it as long as Freaknik comes back every year, but before he can finish speaking, a golden lion statue-shaped ship comes right in, inhabited by the members of the Boule. But as Freaknik is about to disqualify them, a dog-shaped spacecraft called the "Mothership Connection" arrives, killing the Boule and their ship (Note: This scene can only be seen on the uncut version of the special). It is inhabited by three brightly colored aliens who are actually George Clinton, Bootsy Collins, and Gene "King Poo Poo Man" Anderson. They say they have come to take Freaknik because "there are other galaxies that need his powers of positivity", saying that maybe someday he will return and they can "funk it up" once again. Freaknik gives Virgil his gold chain and says that Atlanta will always be his home. Suzie approaches Freaknik, telling him her baby (which looks like Freaknik) needs a father. Freaknik then rushes on board the ship with Clinton, Collins, and Anderson. Mr. Thanksgiving, the radio DJ from the beginning of the show, then speaks, saying how crazy that was and they'll see us next time; Freaknik is seen dancing on the Mothership as it leaves Earth. And after the end credits, we see Sweet Tea taping the check back together.  Voice cast [ edit ]  Additional voices are provided by: Gerald "Slink" Johnson, Heather Lawless, Jason Van Veen, and Jason Walden.  Production [ edit ]  Freaknik: The Musical originally evolved from a failed pilot entitled That Crook'd 'Sipp which was created by Mike Weiss, Jacob Escobedo, and Nick Weidenfeld. The pilot premiered on television on May 13, 2007.[1][2] Originally, the pilot was to receive six additional episodes scheduled to air sometime after 2007, but the episodes never surfaced and the show's status remained up in the air until mid-2009 when the series was scrapped for good in order to create this special.[3][4] Characters including Big Uzi, Suzy and Virgil all appeared in That Crook'd Sipp.  Reception [ edit ]  In its original American broadcast on March 7, 2010, Freaknik: The Musical was watched by 797,000 viewers 18-34, making it the second most watched Adult Swim program of that night, behind a rerun of Family Guy.[5]  IGN gave this episode a 6.1 out of 10, which is considered "Passable", and received comments both positive and negative.[6]  Home release [ edit ]  On March 8, 2010, the animated special was released for purchase on the iTunes Store.[7] The uncut 49-minute-long version of Freaknik: The Musical was released on one-disc DVD set in the United States on October 26, 2010,[8][9] from Warner Home Video and included the soundtrack.  Soundtrack [ edit ]  It was announced by T-Pain that a soundtrack would be released through Jive Records, Konvict Muzik and Nappy Boy on April 20, 2010. The track "Ghetto Commandments", the credits outro song, was released on iTunes as a single on March 23 and it features rappers Snoop Dogg & Mack Maine who also play in the movie; it was released the same day as the release of T-Pain's promo single for his album "rEVOLVEr" "Reverse Cowgirl". The Rick Ross song "Grab Yo Beltloop" didn't make the final cut for the album.  No. Title Producer(s) Length 1. "Freaknik Is Back" T-Pain a.k.a. Ghost of Freaknik ) Tha Bizness 2:26 2. "Save You" T-Pain a.k.a. Ghost of Freaknik featuring One Chance's Jon A. Gordon, Michael A. Gordon) Jon A. Gordon, Michael A. Gordon) Tha Bizness 3:42 3. "Ghetto Commandments" T-Pain a.k.a. Ghost of Freaknik featuring Snoop Dogg & Mack Maine ) Tha Bizness 4:47 4. "We The Mob" T-Pain a.k.a. Ghost of Freaknik featuring Young Cash a.k.a. Virgil ) Ky Miller 3:02 5. "Beat Build" T-Pain a.k.a. Ghost of Freaknik featuring Young Cash a.k.a. Virgil & Rick Ross a.k.a. Big Uzi ) Tha Bizness 3:31"""

In [73]:
bos, eos = tokenizer.bos_token_id, tokenizer.eos_token_id          # both 50256 for GPT-2
inner = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids[0].tolist()
x0 = torch.tensor([[bos, *inner, eos]], device=device)             # [B=1, L]
B, L = x0.shape

# Truncate to max of 1024 tokens
if L > 1024:
    x0 = x0[:, :1024]
    L = 1024
    
print(f"L = {L} tokens:", tokenizer.convert_ids_to_tokens(x0[0].tolist()))

L = 1024 tokens: ['<|endoftext|>', 'Fre', 'ak', 'nik', ':', 'ĠThe', 'ĠMusical', 'Ġis', 'Ġan', 'ĠAmerican', 'Ġmusical', 'Ġtelevision', 'Ġspecial', 'Ġproduced', 'Ġby', 'ĠT', '-', 'Pain', '.', 'ĠIt', 'Ġfeatures', 'Ġthe', 'Ġvoice', 'Ġof', 'ĠT', '-', 'Pain', 'Ġas', 'Ġthe', 'ĠGhost', 'Ġof', 'ĠFreak', 'nik', ',', 'Ġas', 'Ġwell', 'Ġas', 'Ġthe', 'Ġvoices', 'Ġof', 'Ġentertain', 'ers', 'Ġsuch', 'Ġas', 'ĠLil', 'ĠWayne', ',', 'ĠYoung', 'ĠCash', ',', 'ĠSno', 'op', 'ĠDog', 'g', ',', 'ĠSophia', 'ĠFresh', ',', 'Ġand', 'ĠRick', 'ĠRoss', ',', 'Ġand', 'Ġcomedians', 'Ġsuch', 'Ġas', 'ĠAndy', 'ĠSam', 'berg', 'Ġand', 'ĠCharlie', 'ĠMurphy', 'Ġwho', 'Ġprovide', 'Ġadditional', 'Ġvoices', '.', 'ĠIt', 'Ġwas', 'Ġscheduled', 'Ġto', 'Ġair', 'Ġon', 'ĠCartoon', 'ĠNetwork', "'s", 'Ġlate', 'Ġnight', 'Ġprogramming', 'Ġblock', 'ĠAdult', 'ĠSwim', 'Ġsometime', 'Ġin', 'Ġ2009', ',', 'Ġbut', 'Ġafter', 'Ġseveral', 'Ġpush', '-', 'backs', ',', 'Ġit', 'Ġpremiered', 'Ġon', 'ĠMarch', 'Ġ7', ',', 'Ġ2010', '.', 'ĠThe', 'Ġmusical', 'Ġis'

## 3. The unmasking rule

DUEL needs a **deterministic** policy: given the denoiser's current output and the set of masked positions, decide which to unmask next. Determinism is what collapses the sum over $L!$ unmasking orders to a single term (Sec. 3 of the paper).

**Every strategy decomposes into two stages.** Once you see this, the design space becomes a small grid:

1. **Score** — give every position a number; higher = better candidate to unmask now.
2. **Select** — pick positions from those scores: top-$k$, or "everyone above a threshold".

| strategy | score | selector | knob |
|---|---|---|---|
| `greedy` | top-1 log-prob (max confidence) | top-$k$ | `k` |
| `prob_margin` | top-1 minus top-2 (uncertainty aware) | top-$k$ | `k` |
| `left_to_right` | leftmost-first (autoregressive order) | top-$k$ | `k` |
| `confidence_threshold` | top-1 log-prob | every position with score $\geq \log\tau$ | `tau` |

The function returns a boolean mask `[B, L]` of positions to unmask **this step**. New strategies = a new score function or a new selector — the structure makes the design space explicit.

> **Block variant.** The paper restricts each rule to one block of length $L'$ at a time. For our short example one block already covers the sentence, so the block-restricted form reduces to the standard one.

In [ ]:
# ----- SCORE functions: rate each position; higher = better unmask candidate. ---

def score_top1(log_probs: torch.Tensor) -> torch.Tensor:
    """Top-1 log-probability at each position (max confidence)."""
    return log_probs.max(dim=-1).values                                 # [B, L]

def score_margin(log_probs: torch.Tensor) -> torch.Tensor:
    """Gap between top-1 and top-2 log-probs (model's preference strength)."""
    top2 = log_probs.topk(2, dim=-1).values                             # [B, L, 2]
    return top2[..., 0] - top2[..., 1]                                  # [B, L]

def score_position(log_probs: torch.Tensor) -> torch.Tensor:
    """Higher score the further left, so top-k picks the leftmost positions."""
    B, L, _ = log_probs.shape
    return -torch.arange(L, device=log_probs.device).float().expand(B, L)


# ----- SELECT functions: turn scores into a [B, L] bool mask of positions to unmask. -----

def select_topk(score: torch.Tensor, is_masked: torch.Tensor, k: int) -> torch.Tensor:
    """Top-k highest-scoring still-masked positions per row."""
    NEG_INF = torch.finfo(score.dtype).min
    L = is_masked.shape[1]
    score = score.masked_fill(~is_masked, NEG_INF)
    _, idx = score.topk(min(k, L), dim=-1)                              # [B, k']
    out = torch.zeros_like(is_masked)
    out.scatter_(1, idx, True)
    return out & is_masked     # discard positions that weren't actually maskable

def select_threshold(score: torch.Tensor, is_masked: torch.Tensor, tau: float) -> torch.Tensor:
    """Every still-masked position with score >= log(tau); fall back to argmax if none."""
    NEG_INF = torch.finfo(score.dtype).min
    L = is_masked.shape[1]
    qualifies = is_masked & (score >= math.log(tau))                    # [B, L]
    none_qualify = ~qualifies.any(dim=1, keepdim=True)                  # [B, 1]
    fallback = F.one_hot(
        score.masked_fill(~is_masked, NEG_INF).argmax(dim=-1), L
    ).bool()                                                            # [B, L]
    return torch.where(none_qualify, fallback, qualifies)


# ----- Dispatcher: each strategy is one line -- pick a SCORE, apply a SELECT. ---
@torch.no_grad()
def unmasking_rule(log_probs: torch.Tensor,
                   is_masked: torch.Tensor,
                   strategy: str = "greedy",
                   k: int = 1,
                   tau: float = 0.9) -> torch.Tensor:
    """Pick which positions to unmask this step. Strategy = score x select.

    Args:
        log_probs : [B, L, V]  log p_theta(x | x_t).
        is_masked : [B, L]     True at still-masked positions.
        strategy  : 'greedy' | 'prob_margin' | 'left_to_right' | 'confidence_threshold'.
        k         : positions per step (top-k strategies).
        tau       : top-1 prob cutoff (confidence_threshold).

    Returns:
        [B, L] bool mask of positions to unmask now.
    """
    if strategy == "greedy":
        return select_topk(score_top1(log_probs), is_masked, k)
    if strategy == "prob_margin":
        return select_topk(score_margin(log_probs), is_masked, k)
    if strategy == "left_to_right":
        return select_topk(score_position(log_probs), is_masked, k)
    if strategy == "confidence_threshold":
        return select_threshold(score_top1(log_probs), is_masked, tau)
    raise ValueError(f"unknown strategy: {strategy}")

## 4. ELBO perplexity

Under the SUBS parameterization, the MDLM ELBO upper-bounds $-\log p(x_0)$:

$$\mathcal{L}_{\text{ELBO}} = \mathbb{E}_{t \sim U[\epsilon, 1]}\,\mathbb{E}_{x_t \mid x_0}\!\left[\, \tfrac{1}{t} \sum_{i:\, x_t^i = [\text{MASK}]} -\log p_\theta(x_0^i \mid x_t) \,\right].$$

**The 1/t weight is schedule-specific.** It comes from the **log-linear schedule** ($\alpha_t = 1 - (1-\epsilon)\,t$, masking probability $\approx t$) used by MDLM-OWT. Verify in `noise_schedule.py:84-86`. Cosine, log, etc. need different weights.

**Per-token normalization.** Each token is masked w.p. $t$, so $\mathbb{E}_t[(1/t)\,\mathbf{1}[i \text{ masked}]] = 1$ — the expected weight per token is 1. The proper per-token average therefore divides by **all** tokens, not just `num_masked`.

We Monte-Carlo the outer expectations: average NLL over $n$ samples of $(t, \text{mask})$ and exponentiate at the end. Average in NLL space, not PPL space.

In [ ]:
@torch.no_grad()
def elbo_nll_per_token(x0: torch.Tensor, n_mc: int = 256, eps: float = 1e-3) -> float:
    """MC estimate of the per-token MDLM ELBO. Log-linear schedule only."""
    B, L = x0.shape
    nll = torch.zeros(B, device=x0.device)        # running NLL per sequence

    for _ in range(n_mc):
        # Step 1: sample noise level t in (eps, 1] and corrupt x_0 to x_t.
        # Each token is independently replaced by [MASK] with probability t.
        t = torch.rand(B, 1, device=x0.device) * (1 - eps) + eps              # [B, 1]
        mask = torch.rand(B, L, device=x0.device) < t                         # [B, L] bool
        xt = torch.where(mask, torch.full_like(x0, MASK_ID), x0)              # [B, L]

        # Step 2: ask the denoiser for log p_theta(x_0 | x_t) at every position.
        log_p = denoise(xt)                                                   # [B, L, V]
        log_p_x0 = log_p.gather(-1, x0.unsqueeze(-1)).squeeze(-1)             # [B, L]

        # Step 3: per-position NLL, then apply the ELBO weighting.
        nll_per_position = -log_p_x0                                          # [B, L]
        # Linear-schedule weight: 1/t at masked positions, 0 at unmasked positions.
        weight = (1.0 / t) * mask.float()                                     # [B, L]
        # MC sample of -log p(x_0) per sequence (sum over positions only).
        nll += (weight * nll_per_position).sum(dim=-1)                        # [B]

    # Per-token NLL per sequence: divide by (n_mc * L). The 1/t weight makes the
    # expected per-token contribution exactly 1, so dividing by ALL tokens
    # (not num_masked) is correct. Average across the batch at the very end.
    return (nll / (n_mc * L)).mean().item()

## 5. DUEL exact perplexity

Algorithm:

1. Start fully masked: $x_t = ([\text{MASK}], \dots, [\text{MASK}])$.
2. While any position is masked:
   1. Run the denoiser to get $\log p_\theta(\cdot \mid x_t)$ at every position.
   2. `unmasking_rule` returns a set $S$ of positions to unmask now.
   3. Accumulate $\sum_{i \in S} -\log p_\theta(x_0^i \mid x_t)$.
   4. Teacher-force $x_t^i \leftarrow x_0^i$ for $i \in S$.
3. Per-token NLL = total / $L$.

**Why exact, not a bound.** A deterministic policy assigns probability 1 to one unmasking order, collapsing the $L!$-term sum in the MDM likelihood to a single product. Multi-position rules are equally exact: under SUBS the per-position predictions factorize, so unmasking a *set* $S$ contributes $\sum_{i \in S} \log p_\theta(x_0^i \mid x_t)$ correctly.

**Why teacher-force.** We score $\log p_\theta(x_0)$, so each step conditions on the *true* prefix in unmasking order — not on the model's predictions.

**Cost.** Between $\lceil L/k \rceil$ and $L$ forward passes depending on the rule. The production version (`exact_likelihood.py`) adds a KV cache and block-causal attention.

In [ ]:
@torch.no_grad()
def duel_nll_per_token(x0: torch.Tensor, strategy: str = "greedy",
                       k: int = 1, tau: float = 0.9) -> float:
    """Exact per-token NLL of x_0 under DUEL with a deterministic policy.

    Algorithm (one outer loop iteration = one denoiser forward pass):

        Initialize: x_t = ([MASK], ..., [MASK]); running NLL = 0.
        Repeat until no position is masked:
            (a) DENOISE      -- get log p_theta(x_0 | x_t) at every position.
            (b) PICK         -- unmasking_rule chooses positions S to unmask.
            (c) ACCUMULATE   -- NLL += sum_{i in S} -log p_theta(x_0[i] | x_t).
            (d) TEACHER-FORCE -- copy x_0[i] into x_t at i in S; mark unmasked.
        Return NLL / L.
    """
    B, L = x0.shape

    # --- (1) Initialize at maximum noise: every position is [MASK]. ----------
    # We will progressively reveal x_0 in an order chosen by `unmasking_rule`.
    xt = torch.full_like(x0, MASK_ID)
    is_masked = torch.ones_like(x0, dtype=torch.bool)
    nll = torch.zeros(B, device=x0.device)

    while is_masked.any():
        # --- (2) DENOISE: predicted distribution over real tokens at every position. ---
        log_p = denoise(xt)                                             # [B, L, V]

        # --- (3) PICK: deterministic policy returns the next set of positions S. ---
        #         (greedy / prob_margin / left_to_right / confidence_threshold)
        to_unmask = unmasking_rule(log_p, is_masked,
                                   strategy=strategy, k=k, tau=tau)     # [B, L]

        # --- (4) ACCUMULATE: each chosen position contributes -log p(x_0[i] | x_t). ---
        # This is the *exact* contribution under the deterministic policy:
        log_p_x0 = log_p.gather(-1, x0.unsqueeze(-1)).squeeze(-1)       # [B, L]
        nll += -(log_p_x0 * to_unmask.float()).sum(dim=-1)              # [B]

        # --- (5) TEACHER-FORCE: reveal the true token at chosen positions. ----
        xt = torch.where(to_unmask, x0, xt)
        is_masked = is_masked & ~to_unmask

    # --- (6) Per-token NLL (averaged across batch). ------------------------
    return (nll / L).mean().item()

## 6. Side-by-side

ELBO is an *upper bound*; DUEL with any deterministic policy is *exact*. On a single short sentence, both numbers and the ranking across strategies are noisy. To reproduce the paper's tables, run `bash scripts/owt_perplexity/mdlm_elbo.sh` and `bash scripts/owt_perplexity/mdlm_duel.sh`.

In [69]:
elbo_nll = elbo_nll_per_token(x0, n_mc=128)
print(f"ELBO PPL : {math.exp(elbo_nll):12,.2f}   (MC upper bound, n_mc=128)")

configs = [
    ("greedy",               {"k": 1}),
    ("prob_margin",          {"k": 1}),
    ("left_to_right",        {"k": 1}),     # unmask 1 token per step
    ("confidence_threshold", {"tau": 0.9}),
]
for strategy, kwargs in configs:
    duel_nll = duel_nll_per_token(x0, strategy=strategy, **kwargs)
    print(f"DUEL PPL : {math.exp(duel_nll):12,.2f}   (strategy={strategy}, {kwargs})")

/share/kuleshov/gt345/.cache/huggingface/modules/transformers_modules/kuleshov-group/mdlm-owt/d0958fa851335ece6c15260ce0025f030673c0fb/modeling_mdlm.py:397: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
/share/kuleshov/gt345/.cache/huggingface/modules/transformers_modules/kuleshov-group/mdlm-owt/d0958fa851335ece6c15260ce0025f030673c0fb/modeling_mdlm.py:147: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/share/kuleshov/gt345/.cache/huggingface/modules/transformers_modules/kuleshov-group/mdlm-owt/d0958fa851335ece6c15260ce0025f030673c0fb/modeling_mdlm.py:285: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


ELBO PPL :        50.84   (MC upper bound, n_mc=128)
DUEL PPL :        36.85   (strategy=greedy, {'k': 1})
DUEL PPL :        39.33   (strategy=prob_margin, {'k': 1})
DUEL PPL :        33.83   (strategy=left_to_right, {'k': 1})
DUEL PPL :        37.17   (strategy=confidence_threshold, {'tau': 0.9})
